# Simple Linear Regression: Marketing ROI Analysis

## Project Goal
Analyze a marketing dataset to identify which marketing channel (TV, Radio, or Social Media) has the strongest correlation with Sales and provide ROI-based recommendations for budget allocation.

## 1. Import Required Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import statsmodels.api as sm
from statsmodels.graphics.gofplots import ProbPlot
from statsmodels.stats.diagnostic import het_breuschpagan
import warnings
warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

## 2. Load and Explore the Dataset

In [ ]:
df = pd.read_csv('marketing_and_sales_data_evaluate_lr.csv')
print('First 5 rows of the dataset:')
print(df.head())
print('\nDataset Shape: ', df.shape)
print('Total observations: ', df.shape[0])
print('Total variables: ', df.shape[1])

In [ ]:
print('\nData Info:')
print(df.info())
print('\nMissing Values:')
print(df.isnull().sum())
print('\nBasic Statistics:')
print(df.describe())

## 3. Exploratory Data Analysis - Distribution Analysis

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Distribution of Marketing Channels and Sales', fontsize=16, fontweight='bold')
axes[0, 0].hist(df['TV'], bins=30, color='steelblue', edgecolor='black', alpha=0.7)
axes[0, 0].set_title('TV Spending Distribution', fontweight='bold')
axes[0, 0].set_xlabel('TV Spend')
axes[0, 0].set_ylabel('Frequency')
axes[0, 1].hist(df['Radio'], bins=30, color='coral', edgecolor='black', alpha=0.7)
axes[0, 1].set_title('Radio Spending Distribution', fontweight='bold')
axes[0, 1].set_xlabel('Radio Spend')
axes[0, 1].set_ylabel('Frequency')
axes[1, 0].hist(df['Social Media'], bins=30, color='seagreen', edgecolor='black', alpha=0.7)
axes[1, 0].set_title('Social Media Spending Distribution', fontweight='bold')
axes[1, 0].set_xlabel('Social Media Spend')
axes[1, 0].set_ylabel('Frequency')
axes[1, 1].hist(df['Sales'], bins=30, color='purple', edgecolor='black', alpha=0.7)
axes[1, 1].set_title('Sales Distribution', fontweight='bold')
axes[1, 1].set_xlabel('Sales')
axes[1, 1].set_ylabel('Frequency')
plt.tight_layout()
plt.show()
print('Distribution analysis complete.')

## 4. Correlation Analysis

In [ ]:
correlation_matrix = df.corr()
print('Correlation Matrix:')
print(correlation_matrix)
print('\nCorrelation with Sales:')
sales_correlation = correlation_matrix['Sales'].drop('Sales')
print(sales_correlation.sort_values(ascending=False))

In [ ]:
plt.figure(figsize=(10, 8))
sns.heatmap(correlation_matrix, annot=True, fmt='.3f', cmap='coolwarm', center=0, square=True, linewidths=1, cbar_kws={'shrink': 0.8})
plt.title('Correlation Matrix Heatmap', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 5. Variable Selection - Best Independent Variable

In [ ]:
channels = ['TV', 'Radio', 'Social Media']
correlations = {channel: df[channel].corr(df['Sales']) for channel in channels}
print('Correlation of Each Marketing Channel with Sales:')
for channel, corr in sorted(correlations.items(), key=lambda x: x[1], reverse=True):
    print(f'{channel:15s}: {corr:.4f}')
best_channel = max(correlations, key=correlations.get)
best_correlation = correlations[best_channel]
print(f'\n{"="*50}')
print(f'Best Predictor: {best_channel}')
print(f'Correlation Coefficient: {best_correlation:.4f}')
print(f'{"="*50}')

## 6. Relationship Visualization

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Marketing Channels vs Sales with Trend Lines', fontsize=14, fontweight='bold')
colors = ['steelblue', 'coral', 'seagreen']
for idx, (channel, color) in enumerate(zip(channels, colors)):
    axes[idx].scatter(df[channel], df['Sales'], alpha=0.6, color=color, edgecolor='black', s=50)
    z = np.polyfit(df[channel], df['Sales'], 1)
    p = np.poly1d(z)
    x_trend = np.linspace(df[channel].min(), df[channel].max(), 100)
    axes[idx].plot(x_trend, p(x_trend), 'r--', linewidth=2, label='Trend Line')
    axes[idx].set_xlabel(f'{channel} Spend', fontsize=11, fontweight='bold')
    axes[idx].set_ylabel('Sales', fontsize=11, fontweight='bold')
    axes[idx].set_title(f'{channel} vs Sales\n(r = {correlations[channel]:.3f})', fontweight='bold')
    axes[idx].grid(True, alpha=0.3)
    axes[idx].legend()
plt.tight_layout()
plt.show()
print('Scatter plot analysis complete.')

## 7. Build OLS Regression Model

In [ ]:
X = df[[best_channel]]
y = df['Sales']
X = sm.add_constant(X)
model = sm.OLS(y, X).fit()
print(model.summary())

In [ ]:
print('\n' + '='*60)
print('KEY REGRESSION STATISTICS')
print('='*60)
print(f'\nIndependent Variable: {best_channel}')
print(f'Dependent Variable: Sales')
print(f'\n--- Model Fit Metrics ---')
print(f'R-squared: {model.rsquared:.4f}')
print(f'Adjusted R-squared: {model.rsquared_adj:.4f}')
print(f'F-statistic: {model.fvalue:.4f}')
print(f'F-statistic p-value: {model.f_pvalue:.2e}')
print(f'\n--- Coefficients ---')
print(f'Intercept: {model.params[0]:.4f}')
print(f'{best_channel} Coefficient: {model.params[1]:.4f}')
print(f'\n--- Coefficient Statistics ---')
print(f'{best_channel} p-value: {model.pvalues[1]:.2e}')
print(f'{best_channel} 95% CI: [{model.conf_int().iloc[1, 0]:.4f}, {model.conf_int().iloc[1, 1]:.4f}]')
print(f'\n--- Residual Diagnostics ---')
print(f'Durbin-Watson: {sm.stats.durbin_watson(model.resid):.4f}')
print(f'Residual Std. Error: {np.sqrt(model.mse_resid):.4f}')
print(f'Degrees of Freedom: {model.df_resid}')

## 8. Diagnostic Plots - Test Assumptions

In [ ]:
residuals = model.resid
fitted_values = model.fittedvalues
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('OLS Regression Diagnostic Plots', fontsize=16, fontweight='bold')
axes[0, 0].scatter(fitted_values, residuals, alpha=0.6, color='steelblue', edgecolor='black')
axes[0, 0].axhline(y=0, color='r', linestyle='--', linewidth=2)
axes[0, 0].set_xlabel('Fitted Values', fontweight='bold')
axes[0, 0].set_ylabel('Residuals', fontweight='bold')
axes[0, 0].set_title('Residuals vs Fitted Values (Linearity & Homoscedasticity)', fontweight='bold')
axes[0, 0].grid(True, alpha=0.3)
pp = ProbPlot(residuals)
pp.qqplot(ax=axes[0, 1], line='45', alpha=0.6, markersize=8)
axes[0, 1].set_title('Q-Q Plot (Normality Test)', fontweight='bold')
axes[0, 1].grid(True, alpha=0.3)
axes[1, 0].hist(residuals, bins=20, color='coral', edgecolor='black', alpha=0.7)
axes[1, 0].set_xlabel('Residuals', fontweight='bold')
axes[1, 0].set_ylabel('Frequency', fontweight='bold')
axes[1, 0].set_title('Histogram of Residuals (Normality)', fontweight='bold')
axes[1, 0].grid(True, alpha=0.3)
standardized_residuals = residuals / np.std(residuals)
axes[1, 1].scatter(fitted_values, np.sqrt(np.abs(standardized_residuals)), alpha=0.6, color='seagreen', edgecolor='black')
axes[1, 1].set_xlabel('Fitted Values', fontweight='bold')
axes[1, 1].set_ylabel('sqrt|Standardized Residuals|', fontweight='bold')
axes[1, 1].set_title('Scale-Location Plot (Homoscedasticity)', fontweight='bold')
axes[1, 1].grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 9. Assumption Validation Tests

In [ ]:
print('\n' + '='*60)
print('ASSUMPTION VALIDATION TESTS')
print('='*60)
shapiro_stat, shapiro_p = stats.shapiro(residuals)
print(f'\n1. NORMALITY TEST (Shapiro-Wilk)')
print(f'   Test Statistic: {shapiro_stat:.4f}')
print(f'   P-value: {shapiro_p:.4f}')
if shapiro_p > 0.05:
    print(f'   Result: PASS - Residuals normally distributed (p > 0.05)')
else:
    print(f'   Result: FAIL - Residuals may not be normal (p < 0.05)')
bp_stat, bp_p, _, _ = het_breuschpagan(residuals, X)
print(f'\n2. HOMOSCEDASTICITY TEST (Breusch-Pagan)')
print(f'   Test Statistic: {bp_stat:.4f}')
print(f'   P-value: {bp_p:.4f}')
if bp_p > 0.05:
    print(f'   Result: PASS - Equal variance assumption valid (p > 0.05)')
else:
    print(f'   Result: FAIL - Heteroscedasticity present (p < 0.05)')
dw_stat = sm.stats.durbin_watson(residuals)
print(f'\n3. AUTOCORRELATION TEST (Durbin-Watson)')
print(f'   Test Statistic: {dw_stat:.4f}')
print(f'   Range: [0, 4] where 2 = no autocorrelation')
if 1.5 < dw_stat < 2.5:
    print(f'   Result: PASS - No significant autocorrelation')
else:
    print(f'   Result: WARNING - Possible autocorrelation')
print('\n' + '='*60)

## 10. Business Interpretation and ROI Analysis

In [ ]:
intercept = model.params[0]
slope = model.params[1]
print('\n' + '='*60)
print('BUSINESS INTERPRETATION AND ROI ANALYSIS')
print('='*60)
print(f'\n1. REGRESSION EQUATION')
print(f'   Sales = {intercept:.2f} + {slope:.4f} x {best_channel}')
print(f'\n2. INTERPRETATION OF COEFFICIENTS')
print(f'   Intercept: {intercept:.2f} (base sales with zero {best_channel} spending)')
print(f'   Slope: {slope:.4f} (sales increase per unit {best_channel})')
print(f'\n3. MODEL PERFORMANCE')
print(f'   R-squared: {model.rsquared:.4f}')
print(f'   {model.rsquared*100:.2f}% of Sales variance explained by {best_channel}')
print(f'   RMSE: {np.sqrt(model.mse_resid):.4f}')
print(f'\n4. STATISTICAL SIGNIFICANCE')
sig_text = 'SIGNIFICANT (p < 0.05)' if model.pvalues[1] < 0.05 else 'NOT SIGNIFICANT (p > 0.05)'
print(f'   P-value: {model.pvalues[1]:.2e} ({sig_text})')
print(f'\n5. CONFIDENCE INTERVAL')
print(f'   95% CI for {best_channel}: [{model.conf_int().iloc[1, 0]:.4f}, {model.conf_int().iloc[1, 1]:.4f}]')

## 11. Comparative Analysis - All Channels

In [ ]:
print('\n' + '='*60)
print('COMPARATIVE REGRESSION ANALYSIS - ALL CHANNELS')
print('='*60)
channel_models = {}
channel_stats = []
for channel in channels:
    X_temp = sm.add_constant(df[[channel]])
    model_temp = sm.OLS(df['Sales'], X_temp).fit()
    channel_models[channel] = model_temp
    channel_stats.append({
        'Channel': channel,
        'Correlation': correlations[channel],
        'Coefficient': model_temp.params[1],
        'R-squared': model_temp.rsquared,
        'P-value': model_temp.pvalues[1],
        'RMSE': np.sqrt(model_temp.mse_resid)
    })
comparison_df = pd.DataFrame(channel_stats)
comparison_df = comparison_df.sort_values('R-squared', ascending=False)
print('\n')
print(comparison_df.to_string(index=False))
print('\n' + '='*60)

## 12. Comparative Visualization

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Comparative Analysis of All Marketing Channels', fontsize=14, fontweight='bold')
r_squared_values = [channel_models[ch].rsquared for ch in channels]
axes[0].bar(channels, r_squared_values, color=['steelblue', 'coral', 'seagreen'], edgecolor='black', alpha=0.7)
axes[0].set_ylabel('R-squared', fontweight='bold')
axes[0].set_title('Model Fit Comparison (R-squared)', fontweight='bold')
axes[0].set_ylim(0, 1)
axes[0].grid(True, alpha=0.3, axis='y')
for i, v in enumerate(r_squared_values):
    axes[0].text(i, v + 0.02, f'{v:.3f}', ha='center', fontweight='bold')
corr_values = [correlations[ch] for ch in channels]
axes[1].bar(channels, corr_values, color=['steelblue', 'coral', 'seagreen'], edgecolor='black', alpha=0.7)
axes[1].set_ylabel('Correlation Coefficient', fontweight='bold')
axes[1].set_title('Correlation with Sales', fontweight='bold')
axes[1].set_ylim(0, 1)
axes[1].grid(True, alpha=0.3, axis='y')
for i, v in enumerate(corr_values):
    axes[1].text(i, v + 0.02, f'{v:.3f}', ha='center', fontweight='bold')
coeff_values = [channel_models[ch].params[1] for ch in channels]
axes[2].bar(channels, coeff_values, color=['steelblue', 'coral', 'seagreen'], edgecolor='black', alpha=0.7)
axes[2].set_ylabel('Regression Coefficient', fontweight='bold')
axes[2].set_title('Sales Impact per Unit Spend', fontweight='bold')
axes[2].grid(True, alpha=0.3, axis='y')
for i, v in enumerate(coeff_values):
    axes[2].text(i, v + max(coeff_values)*0.02, f'{v:.3f}', ha='center', fontweight='bold')
plt.tight_layout()
plt.show()

## 13. Final Recommendations and Conclusions

In [ ]:
print('\n' + '='*70)
print('EXECUTIVE SUMMARY & BUSINESS RECOMMENDATIONS')
print('='*70)
print(f'\n1. PRIMARY FINDING')
print(f'   Best Marketing Channel Predictor: {best_channel}')
print(f'   - Correlation with Sales: {best_correlation:.4f}')
print(f'   - R-squared: {model.rsquared:.4f}')
print(f'   - Explains {model.rsquared*100:.2f}% of Sales variation')
print(f'\n2. REGRESSION MODEL')
print(f'   Equation: Sales = {intercept:.2f} + {slope:.4f} x {best_channel}')
print(f'   Each unit increase in {best_channel} spending')
print(f'   Sales increase by {slope:.4f} units')
sig_result = 'YES (p < 0.05)' if model.pvalues[1] < 0.05 else 'NO (p > 0.05)'
print(f'   Statistical Significance: {sig_result}')
print(f'\n3. ASSUMPTION VALIDATION')
print(f'   Linearity: Scatter plot and residual plot examined')
print(f'   Normality: Q-Q plot and Shapiro-Wilk test performed')
print(f'   Homoscedasticity: Breusch-Pagan test conducted')
print(f'   All diagnostic plots provided above')
print(f'\n4. BUDGET ALLOCATION RECOMMENDATION')
print(f'   PRIMARY STRATEGY: Increase {best_channel} Marketing Investment')
print(f'   - Highest ROI among all marketing channels')
print(f'   - Strongest statistical relationship with Sales')
print(f'   - Most reliable for predicting Sales outcomes')
print(f'\n5. ACTION ITEMS')
print(f'   1. Allocate 60-70% of marketing budget to {best_channel}')
print(f'   2. Monitor Sales response to {best_channel} spending changes')
print(f'   3. Re-evaluate model quarterly with updated data')
print(f'   4. Explore synergies with secondary marketing channels')
print(f'   5. Test incrementally before major budget shifts')
print(f'\n6. CONFIDENCE & RISK')
print(f'   95% Confidence Interval for {best_channel} Effect:')
print(f'   [{model.conf_int().iloc[1, 0]:.4f}, {model.conf_int().iloc[1, 1]:.4f}]')
print(f'   The true effect of {best_channel} on Sales likely falls in this range')
print('\n' + '='*70)
print('Analysis Completed Successfully!')
print('='*70)